## Merging Datasets and Combining Open Payments Datasets

This script merges the Open Payments datasets together.

This dataset feature engineers and pivots the aforementioned data to make sure they are all at the same grain: year and NPI. 

Careful consideration to joins and `validate="one_to_one"` on every merge was performed to ensure there were no cross products or unintentional duplicates.

**Datasets required:**
- /dsa/groups/casestudycf25/team02/silver/general_payments-sliced_clean.csv (includes product indicators, fraud features, DME flags, and target from upstream)
- /dsa/groups/casestudycf25/team02/ownership_payment_clean.csv

**Feature Engineering and Cleaning:**
- Convert dataset cols to snake case
- Map General Payments No/Yes columns to binary 0 and 1
- Create counts and sum columns for General Payments at the (`covered_recipient_npi`, `program_year`, `applicable_manufacturer_or_applicable_gpo_making_payment_id`) level
- Roll up fraud-oriented features (`is_weekend_payment`, `is_q4_payment`) as sums per provider-year
- Roll up DME binary columns (19 categories: `dme_glucose_monitors`, `dme_urological_supplies`, `dme_surgical_dressings`, etc.) as sums per provider-year
- Create form-of-payment count and sum pivots (cash, stock, in-kind, etc.)
- Create nature-of-payment count and sum pivots (consulting, food & beverage, education, etc.)
- Create third-party recipient count and sum pivots
- Group the Ownership data by (`covered_recipient_npi`, `program_year`, `applicable_manufacturer_or_applicable_gpo_making_payment_id`) and aggregate:
  - `total_invested_usd` = sum of total_amount_invested_us_dollars
  - `total_value_of_interest` = sum of value_of_interest
  - `n_owner_records` = nunique of record_id
- Pivot the Ownership Data by `interest_held_by_physician_or_an_immediate_family_member` to create two new cols
- Left Join the General Payments and the Ownership data (keeping all General Payments) by (`covered_recipient_npi`, `program_year`, `applicable_manufacturer_or_applicable_gpo_making_payment_id`)
- Group Open Payments to yield a dataset at NPI + program year, summing numeric columns and counting unique manufacturing IDs
- Roll up transaction-level `target` to provider-year level

This script outputs interim CSVs as the merges and transformations take place.

**Returns:** `open_payments_npi_year.csv` to the /dsa/groups/casestudycf25/team02/silver/ folder.

In [1]:
# function to convert all datasets to snake_case
import re

def to_snake_case(name: str) -> str:
    # Add underscore between lower-to-upper transitions
    name = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)

    # Replace non-alphanumeric with underscores
    name = re.sub(r'[^0-9a-zA-Z]+', '_', name)

    # Remove leading/trailing underscores and lowercase
    return name.strip("_").lower()

# Loading Data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

### General Payments

### General Payments

**UPDATED**: The product rollup columns (covered_device, covered_drug, etc.) are now
included directly in the `general_payments-sliced_clean.csv` from the upstream
`general_payments_silver` notebook. This eliminates the need to re-read the products
dimension and recreate the pivot here.

The sliced file now contains:
- All transaction-level payment data
- Product indicator columns at record level
- Target column from LEIE merge


In [3]:
########################################
# LOAD GENERAL PAYMENTS WITH PRODUCT ROLLUP
########################################
# Product columns are now included in sliced_clean from upstream general_payments_silver
# No need to re-read products dimension or recreate pivot

merged_gp = pd.read_csv("/dsa/groups/casestudycf25/team02/silver/general_payments-sliced_clean.csv")

print(f"Loaded General Payments: {merged_gp.shape}")

# Verify expected columns are present
expected_product_cols = ['covered_device', 'covered_drug', 'covered_biological', 
                         'covered_medical supply', 'non-covered_device', 
                         'non-covered_drug', 'non-covered_medical supply']

# Check which product columns exist (some may not be present if no records had those types)
product_cols_present = [c for c in merged_gp.columns if 'covered_' in c.lower() or 'non-covered_' in c.lower()]
product_cols_present = [c for c in product_cols_present if c not in [
    'covered_recipient_npi', 'third_party_equals_covered_recipient_indicator'
]]
print(f"Product columns present: {product_cols_present}")

if "target" not in merged_gp.columns:
    raise KeyError("Expected `target` in general_payments-sliced_clean.csv from general_payments_silver.")

merged_gp[["record_id", "covered_recipient_npi", "program_year", "target"]].head()


Loaded General Payments: (932908, 64)
Product columns present: ['covered_biological', 'covered_device', 'covered_drug', 'covered_medical supply', 'non-covered_device', 'non-covered_drug', 'non-covered_medical supply', 'non-covered_unknown']


,record_id,covered_recipient_npi,program_year,target
0,1006679101,1750964185,2023,0
1,1006696505,1417944091,2023,0
2,1006744861,1699208850,2023,0
3,1006725903,1124052717,2023,0
4,1006725915,1124052717,2023,0


In [4]:
########################
# Mapping Text columns to numbers
#############################
mapping = {'No': 0, 'Yes': 1}

In [5]:
merged_gp["charity_indicator"] = (
      merged_gp["charity_indicator"]
      .map(mapping)     # map Yes/No
      .fillna(0)       # turn NaN into 0
      .astype('int8')
)

In [6]:
# merged_gp["physician_ownership_indicator"] = (
#       merged_gp["physician_ownership_indicator"]
#       .map(mapping)     # map Yes/No
#       .fillna(0)       # turn NaN into 0
#       .astype('int8')
# )

In [7]:
merged_gp["third_party_equals_covered_recipient_indicator"]= (
      merged_gp["third_party_equals_covered_recipient_indicator"]
      .map(mapping)     # map Yes/No
      .fillna(0)       # turn NaN into -1
      .astype('int8')
)

In [8]:
merged_gp["dispute_status_for_publication"]= (
      merged_gp["dispute_status_for_publication"]
      .map(mapping)     # map Yes/No
      .fillna(0)       # turn NaN into 0
      .astype('int8')
)

In [9]:
merged_gp.head(1)

,record_id,payment_publication_date,date_of_payment,program_year,covered_recipient_npi,applicable_manufacturer_or_applicable_gpo_making_payment_id,total_amount_of_payment_us_dollars,number_of_payments_included_in_total_amount,form_of_payment_or_transfer_of_value,nature_of_payment_or_transfer_of_value,...,dme_oxygen_equipment,dme_humidifiers_and_nebulizers,dme_wheelchair_access,dme_pharmacy_dispensing,dme_infusion_pumps,dme_inhalation_solutions,dme_breathing_aids,dme_hospital_beds,dme_replacement_batteries,dme_tapes_and_medical_supplies
0,1006679101,2025-06-30,2023-09-29,2023,1750964185,100000010503,10.58,1,In-kind items and services,Food and Beverage,...,0,0,0,0,0,0,0,0,0,0


In [10]:
merged_gp.columns

Index(['record_id', 'payment_publication_date', 'date_of_payment',
       'program_year', 'covered_recipient_npi',
       'applicable_manufacturer_or_applicable_gpo_making_payment_id',
       'total_amount_of_payment_us_dollars',
       'number_of_payments_included_in_total_amount',
       'form_of_payment_or_transfer_of_value',
       'nature_of_payment_or_transfer_of_value', 'nature_short_descr',
       'city_of_travel', 'state_of_travel', 'country_of_travel',
       'physician_ownership_indicator',
       'third_party_payment_recipient_indicator',
       'name_of_third_party_entity_receiving_payment_or_transfer_of_value',
       'charity_indicator', 'third_party_equals_covered_recipient_indicator',
       'dispute_status_for_publication', 'avg_amount_per_payment',
       'log_total_amount', 'payment_month', 'payment_quarter',
       'is_weekend_payment', 'is_q4_payment',
       'payment_to_publication_lag_days', 'is_travel_payment',
       'is_consulting_payment', 'is_speaker_paymen

In [11]:
###############
# Group the General Payments to Program Year, NPI, and Manufacturing ID

# merged_gp['form_of_payment_or_transfer_of_value'].unique() #categorical
# merged_gp['third_party_payment_recipient_indicator'].unique() #categorical
# merged_gp['nature_short_descr']  #categorical
# merged_gp['name_of_third_party_entity_receiving_payment_or_transfer_of_value'].nunique() # unique names
# applicable_manufacturer_or_applicable_gpo_making_payment_id #unique ids
# merged_gp["charity_indicator"] #binary
# merged_gp["third_party_equals_covered_recipient_indicator"] #binary
# merged_gp["dispute_status_for_publication"] #binary

# #counts
# ['covered_biological','covered_device', 'covered_drug', 'covered_medical supply',
#  'non-covered_device', 'non-covered_drug', 'non-covered_medical supply'] 

In [12]:
merge_keys = [
    'covered_recipient_npi',
    'program_year',
    'applicable_manufacturer_or_applicable_gpo_making_payment_id'
]
value_col = 'total_amount_of_payment_us_dollars'

In [13]:
def make_count_and_sum(
    df: pd.DataFrame,
    group_cols,
    cat_col: str,
    value_col: str,
    count_prefix: str,
    sum_prefix: str,
):
    """
    Returns:
      counts_df: one column per category with counts
      sums_df:   one column per category with dollar sums
    """
    tmp = (
        df
        .groupby(group_cols + [cat_col])
        .agg(
            count=('record_id', 'size'),
            total_sum=(value_col, 'sum'),
        )
        .reset_index()
    )

    counts = tmp.pivot_table(
        index=group_cols,
        columns=cat_col,
        values='count',
        fill_value=0
    )

    sums = tmp.pivot_table(
        index=group_cols,
        columns=cat_col,
        values='total_sum',
        fill_value=0.0
    )

    counts = counts.rename(
        columns=lambda c: f"{count_prefix}_{to_snake_case(str(c))}"
    )
    sums = sums.rename(
        columns=lambda c: f"{sum_prefix}_{to_snake_case(str(c))}"
    )

    counts = counts.reset_index()
    sums = sums.reset_index()

    return counts, sums

In [14]:
####################
# Create Count and Sum columns for each of the categorical cols
######################

# Form of payment
form_counts, form_sums = make_count_and_sum(
    merged_gp,
    group_cols=merge_keys,
    cat_col='form_of_payment_or_transfer_of_value',
    value_col=value_col,
    count_prefix='form_count',
    sum_prefix='form_sum'
)

# Third-party payment recipient indicator
tpp_counts, tpp_sums = make_count_and_sum(
    merged_gp,
    group_cols=merge_keys,
    cat_col='third_party_payment_recipient_indicator',
    value_col=value_col,
    count_prefix='third_party_recipient_count',
    sum_prefix='third_party_recipient_sum'
)

# Nature short description
nature_counts, nature_sums = make_count_and_sum(
    merged_gp,
    group_cols=merge_keys,
    cat_col='nature_short_descr',
    value_col=value_col,
    count_prefix='nature_count',
    sum_prefix='nature_sum'
)

In [15]:
third_party_summary = (
    merged_gp
    .groupby(merge_keys)
    .agg(
        n_third_party_entities=(
            'name_of_third_party_entity_receiving_payment_or_transfer_of_value',
            'nunique'
        )
    )
    .reset_index()
)

In [16]:
numeric_gp_cols = [
    
    # binary cols
    'physician_ownership_indicator',
    'charity_indicator',
    'third_party_equals_covered_recipient_indicator',
    'dispute_status_for_publication',
    
    
    # count and monetary cols
    'total_amount_of_payment_us_dollars',
    'number_of_payments_included_in_total_amount',
    'covered_biological',
    'covered_device',
    'covered_drug',
    'covered_medical supply',
    'non-covered_device',
    'non-covered_drug',
    'non-covered_medical supply',
    
    # Weekend and Q4 payment flags (from general_payments_silver)
    'is_weekend_payment',
    'is_q4_payment',
    
    # DME binary columns (from general_payments_silver)
    'dme_glucose_monitors',
    'dme_urological_supplies',
    'dme_surgical_dressings',
    'dme_positive_airway',
    'dme_lower_limb_orthotics',
    'dme_parenteral_nutrition',
    'dme_ventilators',
    'dme_diabetes_supplies_and_contraceptives',
    'dme_oxygen_accessories',
    'dme_oxygen_equipment',
    'dme_humidifiers_and_nebulizers',
    'dme_wheelchair_access',
    'dme_pharmacy_dispensing',
    'dme_infusion_pumps',
    'dme_inhalation_solutions',
    'dme_breathing_aids',
    'dme_hospital_beds',
    'dme_replacement_batteries',
    'dme_tapes_and_medical_supplies',
]

numeric_gp_agg = (
    merged_gp
    .groupby(merge_keys)[numeric_gp_cols]
    .sum()
    .reset_index()
)

In [17]:
record_counts = (
    merged_gp
    .groupby(merge_keys)
    .size()
    .reset_index(name='n_records')
)

In [18]:
geo_agg = (
    merged_gp
    .groupby(merge_keys)
    .agg(
        n_cities=('city_of_travel', 'nunique'),
        n_states=('state_of_travel', 'nunique'),
        n_countries=('country_of_travel', 'nunique'),
    )
    .reset_index()
)

In [19]:
###########
# Drop these columns, unnecessary for general payments agg df
###########
cols_to_drop = [
    'change_type',
    'payment_publication_date', 
    'nature_of_payment_or_transfer_of_value'
]

merged_gp = merged_gp.drop(columns=cols_to_drop, errors='ignore')

In [20]:
dfs_to_merge = [
    record_counts,
    numeric_gp_agg,
    geo_agg,
    form_counts, form_sums,
    tpp_counts, tpp_sums,
    nature_counts, nature_sums,
    third_party_summary
]

In [21]:
# The aggregated dataframes of General Payments from all the pivoting and grouping must be merged! The list is below.
for df in dfs_to_merge:
    print(df.shape)

(327346, 4)
(327346, 37)
(327346, 6)
(327346, 9)
(327346, 9)
(327346, 6)
(327346, 6)
(320630, 13)
(320630, 13)
(327346, 4)


In [22]:
from functools import reduce

# Merge aggregated dataframes of General Payments. 
# Based on 'covered_recipient_npi', 'program_year', 'applicable_manufacturer_or_applicable_gpo_making_payment_id'
gp_summary = reduce(
    lambda left, right: left.merge(right, on=merge_keys, how='left'),
    dfs_to_merge
)

In [23]:
gp_summary.shape

(327346, 80)

In [24]:
# Delete the temp agg tables
del (numeric_gp_agg, 
 geo_agg,
    form_counts, form_sums,
    tpp_counts, tpp_sums,
    nature_counts, nature_sums,
    third_party_summary,
    dfs_to_merge, record_counts)

In [25]:
gp_summary.columns

Index(['covered_recipient_npi', 'program_year',
       'applicable_manufacturer_or_applicable_gpo_making_payment_id',
       'n_records', 'physician_ownership_indicator', 'charity_indicator',
       'third_party_equals_covered_recipient_indicator',
       'dispute_status_for_publication', 'total_amount_of_payment_us_dollars',
       'number_of_payments_included_in_total_amount', 'covered_biological',
       'covered_device', 'covered_drug', 'covered_medical supply',
       'non-covered_device', 'non-covered_drug', 'non-covered_medical supply',
       'is_weekend_payment', 'is_q4_payment', 'dme_glucose_monitors',
       'dme_urological_supplies', 'dme_surgical_dressings',
       'dme_positive_airway', 'dme_lower_limb_orthotics',
       'dme_parenteral_nutrition', 'dme_ventilators',
       'dme_diabetes_supplies_and_contraceptives', 'dme_oxygen_accessories',
       'dme_oxygen_equipment', 'dme_humidifiers_and_nebulizers',
       'dme_wheelchair_access', 'dme_pharmacy_dispensing',
       

In [26]:
gp_summary.head()

,covered_recipient_npi,program_year,applicable_manufacturer_or_applicable_gpo_making_payment_id,n_records,physician_ownership_indicator,charity_indicator,third_party_equals_covered_recipient_indicator,dispute_status_for_publication,total_amount_of_payment_us_dollars,number_of_payments_included_in_total_amount,...,nature_sum_charity,nature_sum_consulting,nature_sum_debt_forgiveness,nature_sum_education,nature_sum_entertainment,nature_sum_faculty_or_speaker,nature_sum_food_and_beverage,nature_sum_other_services,nature_sum_ownership_or_investment,n_third_party_entities
0,1003000597,2021,100000011212,3,NoNoNo,0,0,0,164.85,3,...,0.0,0.0,0.0,0.0,0.0,0.0,164.85,0.0,0.0,0
1,1003000597,2021,100000346809,1,No,0,0,0,80.00,1,...,0.0,0.0,0.0,0.0,0.0,0.0,80.00,0.0,0.0,0
2,1003000597,2022,100000000106,1,No,0,0,0,15.19,1,...,0.0,0.0,0.0,0.0,0.0,0.0,15.19,0.0,0.0,0
3,1003000597,2022,100000005674,3,NoNoNo,0,0,0,67.55,3,...,0.0,0.0,0.0,0.0,0.0,0.0,67.55,0.0,0.0,0
4,1003000597,2022,100000136378,3,NoNoNo,0,0,0,250.74,3,...,0.0,0.0,0.0,0.0,0.0,0.0,250.74,0.0,0.0,0


In [27]:
gp_summary.describe()

,covered_recipient_npi,program_year,applicable_manufacturer_or_applicable_gpo_making_payment_id,n_records,charity_indicator,third_party_equals_covered_recipient_indicator,dispute_status_for_publication,total_amount_of_payment_us_dollars,number_of_payments_included_in_total_amount,covered_biological,...,nature_sum_charity,nature_sum_consulting,nature_sum_debt_forgiveness,nature_sum_education,nature_sum_entertainment,nature_sum_faculty_or_speaker,nature_sum_food_and_beverage,nature_sum_other_services,nature_sum_ownership_or_investment,n_third_party_entities
count,3.273460e+05,327346.000000,3.273460e+05,327346.000000,327346.000000,327346.000000,327346.000000,3.273460e+05,327346.000000,327346.000000,...,320630.000000,3.206300e+05,320630.000000,320630.000000,320630.000000,3.206300e+05,320630.000000,320630.000000,320630.000000,327346.000000
mean,1.500230e+09,2022.100020,1.000001e+11,2.849914,0.000021,0.006846,0.000171,1.550232e+03,2.890743,0.001870,...,0.016798,2.227844e+02,2.269842,53.995339,0.198792,2.805769e+01,104.970899,80.771212,5.726580,0.010432
std,2.881043e+08,0.812251,2.783034e+05,6.172292,0.004624,0.219515,0.026964,9.158566e+04,6.247692,0.113364,...,3.616835,5.777417e+03,411.182760,1178.151741,11.629216,4.898882e+03,258.890833,2281.638428,2209.179859,0.123977
min,1.003001e+09,2021.000000,1.000000e+11,1.000000,0.000000,0.000000,0.000000,7.900000e-01,1.000000,0.000000,...,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
25%,1.245965e+09,2021.000000,1.000000e+11,1.000000,0.000000,0.000000,0.000000,2.004000e+01,1.000000,0.000000,...,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,18.900000,0.000000,0.000000,0.000000
50%,1.508048e+09,2022.000000,1.000000e+11,1.000000,0.000000,0.000000,0.000000,4.550000e+01,1.000000,0.000000,...,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,39.900000,0.000000,0.000000,0.000000
75%,1.750335e+09,2023.000000,1.000000e+11,2.000000,0.000000,0.000000,0.000000,1.357975e+02,2.000000,0.000000,...,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,119.390000,0.000000,0.000000,0.000000
max,1.992998e+09,2023.000000,1.000014e+11,366.000000,1.000000,55.000000,9.000000,2.223778e+07,366.000000,42.000000,...,1546.000000,2.391608e+06,165096.840000,172950.000000,2028.930000,2.520955e+06,58793.730000,507540.000000,1000000.000000,11.000000


In [28]:
# Output general_payments_record_id_level_clean

gp_summary.to_csv("general_payments_npi_year_manufacturing_clean.csv", index = False)

### Ownership Payments

In [29]:
ownership = pd.read_csv("/dsa/groups/casestudycf25/team02/ownership_payment_clean.csv")

In [30]:
ownership.shape

(12710, 15)

In [31]:
ownership.head()

,Change_Type,Physician_Profile_ID,Physician_NPI,Record_ID,Program_Year,Total_Amount_Invested_USDollars,Value_of_Interest,Terms_of_Interest,Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID,Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name,Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_State,Dispute_Status_for_Publication,Interest_Held_by_Physician_or_an_Immediate_Family_Member,Payment_Publication_Date,FULL_NAME
0,UNCHANGED,361157,1.962476e+09,754966674,2021,0.0,296.69,EQUITY DISTRIBUTION PAYMENT,100000011188,"MOBIUS THERAPEUTICS, LLC",MO,No,Physician Covered Recipient,6/30/2025,MARLENE MOSTER
1,UNCHANGED,173155,1.821079e+09,754966682,2021,0.0,593.37,EQUITY DISTRIBUTION PAYMENT,100000011188,"MOBIUS THERAPEUTICS, LLC",MO,No,Physician Covered Recipient,6/30/2025,ROBERT RITCH
2,NEW,84927,1.881709e+09,755268702,2021,0.0,31926.00,PRIVATE EQUITY,100000806858,"MANUAL SURGICAL SCIENCES, LLC",UT,No,Physician Covered Recipient,6/30/2025,ANDRELUIZ DAVILA
3,NEW,119199,1.639176e+09,755300384,2021,0.0,43161.00,PRIVATE EQUITY,100000806858,"MANUAL SURGICAL SCIENCES, LLC",UT,No,Physician Covered Recipient,6/30/2025,SHEPHAL DOSHI
4,NEW,233626,1.144259e+09,755306684,2021,0.0,87984.00,PRIVATE EQUITY,100000806858,"MANUAL SURGICAL SCIENCES, LLC",UT,No,Physician Covered Recipient,6/30/2025,SRINIVAS DUKKIPATI


In [32]:
ownership = ownership.rename(columns={col: to_snake_case(col) for col in ownership.columns})

In [33]:
ownership["interest_held_by_physician_or_an_immediate_family_member"].unique()

array(['Physician Covered Recipient', 'Immediate family member'],
      dtype=object)

In [34]:
ownership["covered_recipient_npi"] = ownership["physician_npi"]

In [35]:
summary = (
    ownership.groupby(['covered_recipient_npi',
                       'program_year',
                       'applicable_manufacturer_or_applicable_gpo_making_payment_id']).agg(
        total_invested_usd=('total_amount_invested_us_dollars', 'sum'),
        total_value_of_interest=('value_of_interest', 'sum'),
        n_owner_records=('record_id', 'nunique')
    )
).reset_index() 

summary.head()

,covered_recipient_npi,program_year,applicable_manufacturer_or_applicable_gpo_making_payment_id,total_invested_usd,total_value_of_interest,n_owner_records
0,1.003001e+09,2021,100000005385,0.0,38419.76,1
1,1.003001e+09,2022,100000005385,0.0,38419.76,1
2,1.003001e+09,2023,100000005385,0.0,38419.76,1
3,1.003013e+09,2021,100000966845,75000.0,169597.00,1
4,1.003013e+09,2022,100000966845,0.0,169590.00,1


In [36]:
summary.columns

Index(['covered_recipient_npi', 'program_year',
       'applicable_manufacturer_or_applicable_gpo_making_payment_id',
       'total_invested_usd', 'total_value_of_interest', 'n_owner_records'],
      dtype='object')

In [37]:
# Pivot interest_held into one column per interest type (counts)
interest_pivot = (
    pd.crosstab(
        [ownership['covered_recipient_npi'], ownership['program_year'], ownership['applicable_manufacturer_or_applicable_gpo_making_payment_id']],
        ownership['interest_held_by_physician_or_an_immediate_family_member']
    )
    .reset_index()
)


interest_table = interest_pivot.groupby(['covered_recipient_npi', 'program_year',
                                         'applicable_manufacturer_or_applicable_gpo_making_payment_id'], 
                                        as_index=False)[['Immediate family member','Physician Covered Recipient']].sum()

interest_table = interest_table.rename(columns={col: to_snake_case(col) for col in interest_table.columns})

interest_table

interest_held_by_physician_or_an_immediate_family_member,covered_recipient_npi,program_year,applicable_manufacturer_or_applicable_gpo_making_payment_id,immediate_family_member,physician_covered_recipient
0,1.003001e+09,2021,100000005385,0,1
1,1.003001e+09,2022,100000005385,0,1
2,1.003001e+09,2023,100000005385,0,1
3,1.003013e+09,2021,100000966845,1,0
4,1.003013e+09,2022,100000966845,1,0
...,...,...,...,...,...
11934,1.992981e+09,2022,100000826852,0,1
11935,1.992981e+09,2023,100000826852,0,1
11936,1.992993e+09,2021,100000151617,0,1
11937,1.992993e+09,2022,100000151617,0,1


In [38]:
# Merge ownership interest pivot with ownership payments summary
ownership_final = summary.merge(
    interest_table,
    on=['covered_recipient_npi', 'program_year', 'applicable_manufacturer_or_applicable_gpo_making_payment_id'],
    how="inner",          
    validate="one_to_one" 
)
ownership_final.head()

,covered_recipient_npi,program_year,applicable_manufacturer_or_applicable_gpo_making_payment_id,total_invested_usd,total_value_of_interest,n_owner_records,immediate_family_member,physician_covered_recipient
0,1.003001e+09,2021,100000005385,0.0,38419.76,1,0,1
1,1.003001e+09,2022,100000005385,0.0,38419.76,1,0,1
2,1.003001e+09,2023,100000005385,0.0,38419.76,1,0,1
3,1.003013e+09,2021,100000966845,75000.0,169597.00,1,1,0
4,1.003013e+09,2022,100000966845,0.0,169590.00,1,1,0


In [39]:
del ownership, interest_pivot, interest_table, summary

### Merging the Payments Tables Together

In [40]:
merge_keys = [
    'covered_recipient_npi',
    'program_year',
    'applicable_manufacturer_or_applicable_gpo_making_payment_id'
]

# In summary
dupes_gp_summary = gp_summary[gp_summary.duplicated(subset=merge_keys, keep=False)] \
    .sort_values(merge_keys)
print("Duplicates in summary:")
print(dupes_gp_summary.head(20))

# In interest_table
dupes_ownership = ownership_final[ownership_final.duplicated(subset=merge_keys, keep=False)] \
    .sort_values(merge_keys)
print("Duplicates in interest_table:")
print(dupes_ownership.head(20))


Duplicates in summary:
Empty DataFrame
Columns: [covered_recipient_npi, program_year, applicable_manufacturer_or_applicable_gpo_making_payment_id, n_records, physician_ownership_indicator, charity_indicator, third_party_equals_covered_recipient_indicator, dispute_status_for_publication, total_amount_of_payment_us_dollars, number_of_payments_included_in_total_amount, covered_biological, covered_device, covered_drug, covered_medical supply, non-covered_device, non-covered_drug, non-covered_medical supply, is_weekend_payment, is_q4_payment, dme_glucose_monitors, dme_urological_supplies, dme_surgical_dressings, dme_positive_airway, dme_lower_limb_orthotics, dme_parenteral_nutrition, dme_ventilators, dme_diabetes_supplies_and_contraceptives, dme_oxygen_accessories, dme_oxygen_equipment, dme_humidifiers_and_nebulizers, dme_wheelchair_access, dme_pharmacy_dispensing, dme_infusion_pumps, dme_inhalation_solutions, dme_breathing_aids, dme_hospital_beds, dme_replacement_batteries, dme_tapes_and_m

In [41]:
# composite key columns
keys = ["covered_recipient_npi", "applicable_manufacturer_or_applicable_gpo_making_payment_id", "program_year"]

# Merging the General Payments final table with the ownership final table
all_payments = gp_summary.merge(
    ownership_final,
    on=keys,
    how="left",         
    validate="one_to_one"
)

In [42]:
all_payments.shape

(327346, 85)

In [43]:
all_payments.to_csv("all_payments_npi_year_manufacturing_clean.csv", index = False)

In [44]:
all_payments.columns

Index(['covered_recipient_npi', 'program_year',
       'applicable_manufacturer_or_applicable_gpo_making_payment_id',
       'n_records', 'physician_ownership_indicator', 'charity_indicator',
       'third_party_equals_covered_recipient_indicator',
       'dispute_status_for_publication', 'total_amount_of_payment_us_dollars',
       'number_of_payments_included_in_total_amount', 'covered_biological',
       'covered_device', 'covered_drug', 'covered_medical supply',
       'non-covered_device', 'non-covered_drug', 'non-covered_medical supply',
       'is_weekend_payment', 'is_q4_payment', 'dme_glucose_monitors',
       'dme_urological_supplies', 'dme_surgical_dressings',
       'dme_positive_airway', 'dme_lower_limb_orthotics',
       'dme_parenteral_nutrition', 'dme_ventilators',
       'dme_diabetes_supplies_and_contraceptives', 'dme_oxygen_accessories',
       'dme_oxygen_equipment', 'dme_humidifiers_and_nebulizers',
       'dme_wheelchair_access', 'dme_pharmacy_dispensing',
       

### Group all Open Payments to NPI + program year

In [45]:
################
# - manufacturing counts
# - sum / mean payment
# - sums for all other numeric columns
################

group_cols = ["covered_recipient_npi", "program_year"]

# All numeric columns
numeric_cols = all_payments.select_dtypes(include="number").columns.tolist()

# Columns we do NOT want to auto-sum
exclude_from_auto_sum = set(
    group_cols
    + [
        "applicable_manufacturer_or_applicable_gpo_making_payment_id",  # get unique counts later
        "total_amount_of_payment_us_dollars",                           # handled separately (sum/mean)
    ]
)

numeric_sum_cols = [c for c in numeric_cols if c not in exclude_from_auto_sum]

# Build dict of sum aggs for the remaining numeric columns
sum_aggs = {c: (c, "sum") for c in numeric_sum_cols}

all_payments_npi_year_final = (
    all_payments
    .groupby(group_cols)
    .agg(
        # manufacturing-related
        n_manufacturing_entities=(
            "applicable_manufacturer_or_applicable_gpo_making_payment_id",
            "nunique"
        ),
        # payment stats
        total_payment=("total_amount_of_payment_us_dollars", "sum"),
        mean_payment_manufacturers =("total_amount_of_payment_us_dollars", "mean"),

        # all other numeric sums
        **sum_aggs,
    )
    .reset_index()
)


In [46]:
all_payments_npi_year_final.to_csv("all_payments_npi_year_final_clean.csv")

In [47]:
all_payments_npi_year_final.shape

(158751, 85)

In [48]:
# Roll the upstream transaction-level target up to provider-year once.
provider_year_target = (
    merged_gp
    .groupby(["covered_recipient_npi", "program_year"])["target"]
    .max()
    .astype("Int64")
    .reset_index()
)

provider_year_target.shape


(158751, 3)

In [49]:
# Attach the binary provider-year target rolled up from the upstream transaction label
final_open_payments = all_payments_npi_year_final.merge(
    provider_year_target,
    on=["covered_recipient_npi", "program_year"],
    how="left",
    validate="one_to_one"
)

# Keep unlabeled provider-years as 0 and preserve a binary integer target
final_open_payments["target"] = final_open_payments["target"].fillna(0).astype("Int64")

final_open_payments.shape


(158751, 86)

In [50]:
final_open_payments.head(1)


,covered_recipient_npi,program_year,n_manufacturing_entities,total_payment,mean_payment_manufacturers,n_records,charity_indicator,third_party_equals_covered_recipient_indicator,dispute_status_for_publication,number_of_payments_included_in_total_amount,...,nature_sum_food_and_beverage,nature_sum_other_services,nature_sum_ownership_or_investment,n_third_party_entities,total_invested_usd,total_value_of_interest,n_owner_records,immediate_family_member,physician_covered_recipient,target
0,1003000597,2021,2,244.85,122.425,4,0,0,0,4,...,244.85,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0


In [51]:
final_open_payments["target"].value_counts(dropna=False).sort_index()


target
0    158709
1        42
Name: count, dtype: Int64

In [52]:
final_open_payments.shape


(158751, 86)

In [53]:
final_open_payments.head()


,covered_recipient_npi,program_year,n_manufacturing_entities,total_payment,mean_payment_manufacturers,n_records,charity_indicator,third_party_equals_covered_recipient_indicator,dispute_status_for_publication,number_of_payments_included_in_total_amount,...,nature_sum_food_and_beverage,nature_sum_other_services,nature_sum_ownership_or_investment,n_third_party_entities,total_invested_usd,total_value_of_interest,n_owner_records,immediate_family_member,physician_covered_recipient,target
0,1003000597,2021,2,244.85,122.425000,4,0,0,0,4,...,244.85,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0
1,1003000597,2022,5,1160.21,232.042000,17,0,0,0,17,...,733.01,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0
2,1003000597,2023,9,971.48,107.942222,30,0,0,0,30,...,971.48,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0
3,1003000902,2021,1,13.50,13.500000,1,0,0,0,1,...,13.50,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0
4,1003000902,2022,1,12.15,12.150000,1,0,0,0,1,...,12.15,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0


In [54]:
print(f"Final dataset shape: {final_open_payments.shape}")
print("Provider-year target remains binary and is derived from transaction-level target.")


Final dataset shape: (158751, 86)
Provider-year target remains binary and is derived from transaction-level target.


In [55]:
final_open_payments.shape

(158751, 86)

In [56]:
final_open_payments.head()


,covered_recipient_npi,program_year,n_manufacturing_entities,total_payment,mean_payment_manufacturers,n_records,charity_indicator,third_party_equals_covered_recipient_indicator,dispute_status_for_publication,number_of_payments_included_in_total_amount,...,nature_sum_food_and_beverage,nature_sum_other_services,nature_sum_ownership_or_investment,n_third_party_entities,total_invested_usd,total_value_of_interest,n_owner_records,immediate_family_member,physician_covered_recipient,target
0,1003000597,2021,2,244.85,122.425000,4,0,0,0,4,...,244.85,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0
1,1003000597,2022,5,1160.21,232.042000,17,0,0,0,17,...,733.01,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0
2,1003000597,2023,9,971.48,107.942222,30,0,0,0,30,...,971.48,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0
3,1003000902,2021,1,13.50,13.500000,1,0,0,0,1,...,13.50,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0
4,1003000902,2022,1,12.15,12.150000,1,0,0,0,1,...,12.15,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0


In [57]:
# Keep the same final CSV name and path; add the provider-year target column
final_open_payments.to_csv("/dsa/groups/casestudycf25/team02/silver/open_payments_npi_year.csv", index=False)


In [58]:
# Check the newly created CSV and confirm the target column is present
temp = pd.read_csv("/dsa/groups/casestudycf25/team02/silver/open_payments_npi_year.csv")

temp.shape, "target" in temp.columns


((158751, 86), True)

In [59]:
del temp